In [2]:
import pandas as pd
import pyterrier as pt
from pathlib import Path

In [3]:
# Sessions laden
sessions_df = pd.read_csv("data/sessions.csv")

print("Geladene Sessions:", len(sessions_df))

Geladene Sessions: 381


In [4]:
# PyTerrier initialisieren und BM25-Retriever laden
if not pt.java.started():
    pt.java.init()

index_path = str(Path.cwd() / "Index_Longeval_SCI_snapshot3")
index = pt.IndexFactory.of(index_path)

bm25 = pt.terrier.Retriever(index, wmodel="BM25", metadata=["docno", "text"])

Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


12:12:45.893 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1,3 GiB of memory would be required.


In [ ]:
# BM25-Retrieval: Top-103 Treffer pro Session
# Positionen 0-2 = pseudo-relevant, 99-102 = nicht-relevant
import re

TOP_K = 103
REL_N = 3
NONREL_START = 99
NONREL_END = 103

# Terrier reagiert auf Sonderzeichen mit Parse-Fehlern → Bereinigung nötig
def clean_for_terrier(q: str) -> str:
    if q is None:
        return ""
    q = str(q)
    q = q.replace(";", " ")
    q = q.replace("'", " ")
    q = q.replace("’", " ")
    q = q.replace("?", " ")
    q = re.sub(r"[\^]", " ", q)
    q = re.sub(r'["\\]', " ", q)
    q = re.sub(r"[{}\[\]\(\)]", " ", q)
    q = re.sub(r"[:]", " ", q)
    q = re.sub(r"[+\-!~*?]", " ", q)
    q = re.sub(r"[#@$%&|/]", " ", q)
    q = re.sub(r"[<>]", " ", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q

rows = []

for _, s in sessions_df.iterrows():
    sid = s["session_id"]
    query_text = clean_for_terrier(s["first_query"])

    res = bm25.search(query_text).head(TOP_K)

    if len(res) < NONREL_END:
        print(f"Query {sid}: nur {len(res)} Treffer, uebersprungen")
        continue

    docnos = res["docno"].astype(str).tolist()
    texts = res["text"].astype(str).tolist()

    rows.append({
        "session_id": sid,
        "doc_variant": "rel",
        "query_text": query_text,
        "docnos": docnos[:REL_N],
        "texts": texts[:REL_N],
    })

    rows.append({
        "session_id": sid,
        "doc_variant": "non_rel",
        "query_text": query_text,
        "docnos": docnos[NONREL_START:NONREL_END],
        "texts": texts[NONREL_START:NONREL_END],
    })

retrieval_df = pd.DataFrame(rows)
retrieval_df.to_csv("data/retrieval_bm25_topic.csv", index=False)

print("Anzahl Retrieval-Zeilen:", len(retrieval_df))

In [8]:
# Manuelle Topics für Sessions, die beim BM25-Retrieval zu wenige Treffer hatten
# (z.B. zu kurze oder seltene Queries wie "obezite", "as-ack").
# Diese Einträge werden direkt an die bestehenden JSONL-Run-Dateien angehängt.
import json
import os

manual_entries = [
    {
        "qid": "547f504ff3c897aa262396385d4af920",
        "query": "yoshua bengio",
        "description": "Documents related to the work and contributions of Yoshua Bengio, a prominent researcher in artificial intelligence and deep learning.",
        "narrative": "Relevant documents include research papers authored or co-authored by Yoshua Bengio, publications on deep learning and neural networks, and citations of his academic impact. Documents unrelated to Bengio or deep learning are not relevant."
    },
    {
        "qid": "e777fb4e6e6a57942bf45de4eb297a56",
        "query": "kondraske",
        "description": "Documents related to research or work associated with Kondraske.",
        "narrative": "Relevant documents include academic publications or research associated with Kondraske and references in scientific or technical literature. Documents unrelated to this researcher are not relevant."
    },
    {
        "qid": "a76f803a54168ee973b5efb8ec3eff57",
        "query": "as-ack",
        "description": "Documents related to the AS-ACK protocol or acknowledgment mechanism in network communication.",
        "narrative": "Relevant documents include technical descriptions of the AS-ACK protocol and research on acknowledgment mechanisms. Documents on unrelated network topics are not relevant."
    },
    {
        "qid": "a681a45c56438f13b7dfeca913f133e3",
        "query": "cervids",
        "description": "Documents related to cervids, the family of mammals including deer, elk, and moose.",
        "narrative": "Relevant documents include biological and ecological research on cervid species and conservation efforts. Documents unrelated to cervids are not relevant."
    },
    {
        "qid": "7943fb88b63cdf0a8aaff0b67e709cd6",
        "query": "obezite",
        "description": "Documents related to obesity research, particularly from Turkish-language literature.",
        "narrative": "Relevant documents include medical research on obesity causes, treatment, and prevention. Documents unrelated to obesity are not relevant."
    }
]

os.makedirs("runfiles", exist_ok=True)

for variant in ["rel", "non_rel"]:
    path = f"runfiles/openai_first_{variant}_tfidf5.jsonl"
    with open(path, "a", encoding="utf-8") as f:
        for entry in manual_entries:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    print(f"Hinzugefuegt zu {path}")

for variant in ["rel", "non_rel"]:
    path = f"runfiles/openai_first_{variant}_tfidf5.jsonl"
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print(f"{path}: {len(lines)} Eintraege")